# Referencial list (array) vs compact array
In computer, a memory is a sequenctial arragement of bits. They are arranged in sequence of 8 bits (byte). Each byte has a memory address to identify it.

The address value is usualy of size 64 bit (8 bytes)

Array or list in general store values in sequencial memory spaces. So given the starting address of array (memory address of first element)  we can calculate the address of any element, and access it directly (random access).

Compact arrays store actual values in the array sequence, whereas referencial lists store the addresses of values in the sequence.

`list` in pyton is referencial list. so to store 10 ints (each of say 8 byte) we need 10 * (8 + 8) bytes = 160 bytes instead of 80 bytes. So its very memory inefficient. Its also an hinderence to caching, as for each subsequent element, the interpreter needs to go to another address to grab the element.

Python does allow you to create compact arrays with the `array` module.

The array module does not provide support for making compact arrays of user-defined data types. Compact arrays of such structures can be created with the lower-level support of a module named ctypes.

`array.array` is strictly limited to basic C built-in data types and cannot accept Python classes like str, dict, or custom user-defined objects.

| Code | C Data Type | Typical Number of Bytes |
|---|---|---:|
| `'b'` | signed char | 1 |
| `'B'` | unsigned char | 1 |
| `'u'` | Unicode char | 2 or 4 |
| `'h'` | signed short int | 2 |
| `'H'` | unsigned short int | 2 |
| `'i'` | signed int | 2 or 4 |
| `'I'` | unsigned int | 2 or 4 |
| `'l'` | signed long int | 4 |
| `'L'` | unsigned long int | 4 |
| `'f'` | float | 4 |
| `'d'` | float | 8 |


In [2]:
from array import array
from pympler import asizeof

l = [1, 2, 3, 4]
a = array('i', [1,2,3,4])
print('Referencial list size:', asizeof.asizeof(l))
print('Cmpact array size:',asizeof.asizeof(a))

Referencial list size: 216
Cmpact array size: 96


## Dynamic array

In [11]:
import ctypes

class DynamicArray:
    def __make_array(self, c: int):
        return (c * ctypes.py_object)() 

    def __resize(self):
        new_arr = self.__make_array(self._capacity * 2)
        for i in range(self._n):
            new_arr[i] = self._A[i]
        self._capacity *= 2
        self._A = new_arr
            
    def __init__(self):
        self._n = 0
        self._capacity = 8
        self._A = self.__make_array(self._capacity)


    def __len__(self):
        return self._n

    def __getitem__(self, n):
        if 0 <= n < self._n:
            return self._A[n]
        else:
            raise IndexError('invalid index')

    def append(self, v):
        if self._n == self._capacity:
            self.__resize()
        self._A[self._n] = v
        self._n += 1

    def __str__(self):
        return " ".join([str(self._A[i]) for i in range(self._n)])

In [19]:
a = DynamicArray()
print(a)
print(a._n)
print(a._capacity)

a.append(22)
print(a)
print(a._n)
print(a._capacity)

a.append(99)
print(a)
print(a._n)
print(a._capacity)

a.append(1111)
print(a)
print(a._n)
print(a._capacity)

for i in range(6):
    a.append(i)
print(a)
print(a._n)
print(a._capacity)



0
8
22
1
8
22 99
2
8
22 99 1111
3
8
22 99 1111 0 1 2 3 4 5
9
16


## Circular array

A circular array is an array that is treated as if its end connects back to its beginning. Instead of stopping at the last element, you "wrap around" to the first element.

So for [A B C D], after the last element D, comes first element A

The wrapping can be done using the % operator.

`effective_index = actual_index % % length_of_array`

To move one step forward: `i = (i + 1) % n`
To move one step backward: `i = (i - 1 + n) % n`


In [9]:
class CircularArray:
    def __init__(self, size = 8, fill = None):
        self._size = size 
        self._l =  [None] * size
        self._seek = 0

    def append(self, v):
        i = self._seek % self._size
        self._l[i] = v 
        self._seek = i+1

    def __str__(self):
        return str(self._l)

In [10]:
a = CircularArray(size=4)
print(a)
for i in range(4):
    a.append(i)
print(a)
a.append(99)
print(a)

[None, None, None, None]
[0, 1, 2, 3]
[99, 1, 2, 3]
